# 📓 Notebook 8 — Valutazione dei modelli, Cross-Validation e Tuning degli iperparametri

Bentornato! 👋 Sei arrivato all'ultimo notebook del corso. Ottimo lavoro! 🎉

## 🎯 Cosa imparerai qui
1. Perché un singolo `train_test_split` non basta
2. **Cross-Validation (K-Fold)**: la tecnica corretta per valutare un modello
3. **GridSearchCV**: come trovare automaticamente i migliori iperparametri
4. **Pipeline**: come incatenare preprocessing + modello senza errori
5. Una **roadmap pratica** per continuare a imparare dopo questo corso

## 🤔 Perché questo notebook è importante
Finora abbiamo costruito modelli, ma li abbiamo valutati "al volo". In un progetto serio (uno stage, un lavoro, una competizione Kaggle) la valutazione **è la metà del lavoro**. Un modello mal valutato può sembrare bellissimo... e poi crollare nel mondo reale.

In [ ]:
# ==========================================
# IMPORT DELLE LIBRERIE BASE
# ==========================================

# Importiamo NumPy, una libreria fondamentale per il calcolo scientifico in Python.
# Gestisce array multidimensionali e operazioni matematiche ad alte prestazioni.
import numpy as np

# Importiamo Pandas, utilizzata per la manipolazione e l'analisi dei dati.
# Permette di lavorare facilmente con strutture dati simili a tabelle (DataFrame).
import pandas as pd

# Importiamo il modulo pyplot di Matplotlib.
# È la libreria principale per creare grafici e visualizzazioni dei dati.
import matplotlib.pyplot as plt

# ==========================================
# IMPORT DEL DATASET
# ==========================================

# Importiamo una funzione che carica il "Breast Cancer Wisconsin dataset".
# È un dataset classico per problemi di classificazione binaria (tumore maligno vs benigno).
from sklearn.datasets import load_breast_cancer

# ==========================================
# IMPORT DEI MODELLI DI MACHINE LEARNING
# ==========================================

# Regressione Logistica: un modello statistico lineare usato per la classificazione.
from sklearn.linear_model import LogisticRegression

# Random Forest: un modello "ensemble" (basato su più alberi decisionali)
# molto robusto e flessibile per classificazione e regressione.
from sklearn.ensemble import RandomForestClassifier

# Support Vector Classifier (SVC): un modello che cerca di trovare l'iperpiano
# migliore per separare le diverse classi di dati.
from sklearn.svm import SVC

# ==========================================
# STRUMENTI DI VALUTAZIONE E PREPARAZIONE
# ==========================================

from sklearn.model_selection import (
    train_test_split,  # Divide il dataset in due parti: una per l'addestramento (train) e una per il test.
    cross_val_score,   # Calcola il punteggio del modello utilizzando la Cross-Validation (convalida incrociata).
    KFold,             # Divide i dati in 'K' parti uguali per la convalida incrociata standard.
    StratifiedKFold,   # Come KFold, ma mantiene la proporzione delle classi in ogni "fetta" (ideale per dataset sbilanciati).
    GridSearchCV       # Cerca automaticamente la combinazione migliore di iperparametri per un modello.
)

# StandardScaler standardizza le feature (le colonne dei dati) rimuovendo la media
# e ridimensionandole in modo che abbiano varianza unitaria. Molti modelli (come SVC) lo richiedono.
from sklearn.preprocessing import StandardScaler

# Pipeline permette di concatenare più passaggi (es. prima StandardScaler, poi SVC)
# in un unico oggetto, semplificando il codice e prevenendo errori (come il data leakage).
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,        # Calcola l'accuratezza (percentuale di predizioni corrette).
    classification_report  # Genera un report testuale con metriche chiave come precision, recall e f1-score.
)

# ==========================================
# IMPOSTAZIONI INIZIALI
# ==========================================

# Fissiamo il "seed" (seme) per il generatore di numeri casuali di NumPy.
# Questo garantisce la riproducibilità: ogni volta che eseguirai il codice,
# i risultati delle operazioni casuali (come la divisione dei dati) saranno identici.
np.random.seed(42)

# Messaggio di conferma per verificare che la cella sia stata eseguita senza errori.
print('Tutto pronto! ✅')

## 1. Il problema del singolo split

Nei notebook precedenti facevamo così:
```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
```

Il problema? **Il risultato dipende molto da come sono stati estratti quei dati di test.** Se sei fortunato hai un test "facile" e il modello sembra fortissimo. Se sei sfortunato hai un test "difficile" e il modello sembra scarso. Stesso modello, valutazioni diverse!
![problem](Media/08/problem.png)

Verifichiamolo concretamente. 👇

In [ ]:
# ==========================================
# CARICAMENTO DEI DATI
# ==========================================

# Carichiamo il dataset "Breast Cancer Wisconsin" preallineato in scikit-learn.
data = load_breast_cancer()

# Separiamo le variabili predittive (X, le caratteristiche geometriche delle cellule)
# dalla variabile target (y, l'etichetta che indica se il tumore è maligno o benigno).
X, y = data.data, data.target

# ==========================================
# ESPERIMENTO DI SPLIT ITERATO (10 VOLTE)
# ==========================================

# Inizializziamo una lista vuota per memorizzare l'accuratezza ottenuta in ogni ciclo.
accuratezze = []

# Avviamo un ciclo for che girerà 10 volte. La variabile 'seed' assumerà i valori da 0 a 9.
for seed in range(10):

    # Dividiamo il dataset in Train (80%) e Test (20%).
    # Cambiando il 'random_state=seed' a ogni ciclo, i dati verranno mescolati e divisi
    # in modo sempre diverso. Il modello vedrà e testerà dati differenti ogni volta.
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)

    # Creiamo un'istanza del modello di Regressione Logistica.
    # Portiamo 'max_iter' a 10000 per dare all'algoritmo abbastanza passaggi (epoche)
    # per raggiungere la convergenza matematica, evitando avvisi di errore (warning).
    modello = LogisticRegression(max_iter=10000)

    # Addestriamo il modello sul set di train specifico di questo ciclo.
    modello.fit(X_tr, y_tr)

    # Valutiamo l'accuratezza del modello sul set di test (dati mai visti in addestramento).
    acc = modello.score(X_te, y_te)

    # Aggiungiamo il punteggio ottenuto alla nostra lista.
    accuratezze.append(acc)

    # Stampiamo il risultato del ciclo corrente formattando l'accuratezza a 4 cifre decimali.
    print(f'Seed {seed}: accuratezza = {acc:.4f}')

# ==========================================
# ANALISI DEI RISULTATI
# ==========================================

# Visualizziamo l'accuratezza peggiore (minimo) e migliore (massimo) registrate nei 10 tentativi.
print(f'\nMinimo: {min(accuratezze):.4f}')
print(f'Massimo: {max(accuratezze):.4f}')

# Calcoliamo la variazione (differenza tra il valore massimo e quello minimo).
# Questa differenza dimostra graficamente l'instabilità del singolo train_test_split:
# le performance del modello sembrano cambiare solo perché abbiamo pescato dati diversi!
print(f'Differenza: {max(accuratezze) - min(accuratezze):.4f}  ← ecco il problema!')

Vedi? Lo stesso modello, sugli stessi dati, può dare valori che variano di alcuni punti percentuali a seconda di come spezzi i dati. 😱

**Soluzione: Cross-Validation.**

## 2. K-Fold Cross-Validation

L'idea è semplice: invece di fare **un solo** split, ne facciamo **K** (di solito K=5 o K=10).

Esempio con K=5:
1. Dividiamo i dati in 5 parti uguali ("fold")
2. Alleniamo il modello su 4 parti, testiamo sulla 5ª → otteniamo `acc_1`
3. Alleniamo su altre 4 (cambiando quale tieni fuori), testiamo sulla rimanente → `acc_2`
4. ... ripetiamo finché ogni fold è stato "il test" una volta
5. La performance finale è la **media** delle 5 accuratezze (con la deviazione standard)

Così ogni esempio viene usato sia per allenare sia per testare, ma **mai contemporaneamente**. Risultato molto più robusto!

```
Fold 1:  [TEST ][train][train][train][train]
Fold 2:  [train][TEST ][train][train][train]
Fold 3:  [train][train][TEST ][train][train]
Fold 4:  [train][train][train][TEST ][train]
Fold 5:  [train][train][train][train][TEST ]
```
![kfold](Media/08/kfold.png)


In [ ]:
# ==========================================
# INIZIALIZZAZIONE DEL MODELLO
# ==========================================

# Creiamo l'istanza del modello di Regressione Logistica.
# Manteniamo 'max_iter=10000' per garantire la convergenza dell'algoritmo su tutti i fold.
modello = LogisticRegression(max_iter=10000)

# ==========================================
# ESECUZIONE DELLA CROSS-VALIDATION
# ==========================================

# Utilizziamo 'cross_val_score' per automatizzare l'intero processo di Cross-Validation.
# Passiamo come argomenti:
#  - modello: l'algoritmo da addestrare e testare
#  - X e y: l'intero dataset (la funzione si occuperà internamente di dividerlo)
#  - cv=5: specifica il numero di "fold" (K=5). Il dataset viene diviso in 5 parti;
#          il modello viene addestrato 5 volte, ogni volta usando 4 parti per il train e 1 per il test.
#  - scoring='accuracy': indica che vogliamo calcolare l'accuratezza per ogni singolo ciclo.
scores = cross_val_score(modello, X, y, cv=5, scoring='accuracy')

# ==========================================
# STAMPA E ANALISI DEI RISULTATI
# ==========================================

# Mostriamo l'array contenente i 5 punteggi di accuratezza ottenuti nei 5 cicli distinti.
print(f'Accuratezze sui 5 fold: {scores}')

# Calcoliamo la media aritmetica dei 5 punteggi.
# Rappresenta la stima reale e robusta delle prestazioni generali del modello.
print(f'Media: {scores.mean():.4f}')

# Calcoliamo la deviazione standard dei punteggi.
# Ci dice quanto i risultati variano tra un fold e l'altro: un valore basso indica
# che il modello è stabile e non dipende da come sono stati pescati i dati.
print(f'Deviazione standard: {scores.std():.4f}')

# Stampiamo il risultato finale nel formato standard scientifico: "Media ± Deviazione Standard".
# Questo risolve il problema del singolo train_test_split, fornendo un voto finale
# realistico e comprensivo dell'incertezza del modello.
print(f'\n→ Risultato finale: {scores.mean():.4f} ± {scores.std():.4f}')

💡 **Come si legge `0.95 ± 0.02`?**
Significa: "il modello ha circa il 95% di accuratezza, con un'oscillazione tipica del 2%". Molto più informativo di un singolo numero!

### StratifiedKFold (per classificazione sbilanciata)
Se hai un dataset sbilanciato (es. 90% classe A, 10% classe B), il normale K-Fold potrebbe pescare un fold con 0 esempi della classe B! La **stratificazione** assicura che ogni fold mantenga le stesse proporzioni dell'intero dataset.

In [ ]:
# ==========================================
# CONFIGURAZIONE DELLA CROSS-VALIDATION STRATIFICATA
# ==========================================

# Creiamo un oggetto 'StratifiedKFold'.
# A differenza del KFold classico, questo metodo è caldamente raccomandato per la classificazione perché:
#  1. Mantiene la stessa proporzione delle classi (es. 60% sani e 40% malati) in ogni singolo fold.
#  2. Evita che un fold si ritrovi, per puro caso, con soli esempi di una classe, falsando la valutazione.
# Parametri impostati:
#  - n_splits=5: dividiamo i dati in 5 blocchi (K=5).
#  - shuffle=True: mescola i dati prima di dividerli. È fondamentale se il dataset originario
#    è ordinato in qualche modo (es. tutti i malati all'inizio e tutti i sani alla fine).
#  - random_state=42: fissa il seed per il mescolamento, garantendo che i fold siano identici
#    ogni volta che si esegue il codice (riproducibilità).
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ==========================================
# CALCOLO DEI PUNTEGGI
# ==========================================

# Eseguiamo la cross-validation passandogli il modello, le feature (X) e il target (y).
# Nota: invece di passare un semplice numero a 'cv' (es. cv=5), passiamo l'oggetto 'skf'
# appena creato per forzare l'uso della strategia stratificata con mescolamento.
scores = cross_val_score(modello, X, y, cv=skf, scoring='accuracy')

# ==========================================
# STAMPA DEI RISULTATI
# ==========================================

# Stampiamo l'array con i 5 punteggi di accuratezza ottenuti nei fold stratificati.
print(f'Accuratezze stratificate: {scores}')

# Stampiamo direttamente il risultato finale combinando la Media e la Deviazione Standard.
# Questo ci dà la misura definitiva e più affidabile di come il modello performa sul dataset.
print(f'Media: {scores.mean():.4f} ± {scores.std():.4f}')

### 🔧 PROVA TU
Confronta tre modelli diversi tramite cross-validation. Chi vince?

In [ ]:
# ==========================================
# DEFINIZIONE DEL DIZIONARIO DEI MODELLI
# ==========================================

# Creiamo un dizionario per contenere i diversi algoritmi che vogliamo testare.
# Questa è una struttura ideale (best practice): associa un nome leggibile (la chiave)
# direttamente all'oggetto del modello configurato (il valore), rendendo il codice scalabile.
modelli = {
    # 1. Regressione Logistica: modello lineare, ottimo come baseline.
    'Logistic Regression': LogisticRegression(max_iter=10000),

    # 2. Random Forest: modello basato su un "insieme" di 100 alberi decisionali (n_estimators).
    # Fissiamo il random_state per rendere riproducibile la scelta casuale delle feature e dei dati.
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),

    # 3. Support Vector Machine (SVM): in questo caso usiamo il classificatore SVC con i parametri di default.
    'SVM': SVC()
}

# ==========================================
# CICLO DI CONFRONTO E VALUTAZIONE
# ==========================================

# Avviamo un ciclo for usando il metodo '.items()' del dizionario.
# A ogni iterazione, la variabile 'nome' prenderà la chiave (la stringa) e
# la variabile 'm' prenderà il valore (l'istanza del modello pronto da addestrare).
for nome, m in modelli.items():

    # Eseguiamo la Cross-Validation a 5 fold sul modello corrente 'm'.
    # In questo modo, testiamo tutti e tre i modelli esattamente sulle stesse identiche condizioni.
    scores = cross_val_score(m, X, y, cv=5, scoring='accuracy')

    # Stampiamo il report finale del modello corrente.
    # Nota di formattazione: '{nome:25s}' riserva uno spazio fisso di 25 caratteri per la stringa del nome.
    # Questo trucco serve ad allineare perfettamente i risultati in colonna, creando una tabella pulita nel terminale.
    print(f'{nome:25s}: {scores.mean():.4f} ± {scores.std():.4f}')

## 3. Iperparametri: cosa sono?

Quando crei un modello, ci sono **due tipi di parametri**:

| Tipo | Chi li sceglie | Esempio |
|------|----------------|---------|
| **Parametri** | Il modello li impara dai dati durante `fit()` | i pesi `w` di una regressione lineare |
| **Iperparametri** | **TU** li scegli prima di chiamare `fit()` | `max_depth` di un albero, `C` di una SVM |

Gli iperparametri controllano *come* il modello impara. Sceglierli male può fare la differenza tra un modello eccezionale e uno mediocre.

**Domanda:** come trovo i valori migliori? Provandoli tutti! Ma in modo intelligente. → **GridSearchCV**.

## 4. GridSearchCV: la ricerca automatica

L'idea: dichiari quali iperparametri vuoi provare e con quali valori, e sklearn:
1. Prova **tutte** le combinazioni
2. Per ognuna fa cross-validation
3. Ti dice qual è la combinazione migliore

Esempio: per un Random Forest vogliamo provare diversi `n_estimators` e `max_depth`.


![grid](Media/08/grid.png)


In [ ]:
# ==========================================
# DEFINIZIONE DELLO SPAZIO DI RICERCA (GRID)
# ==========================================

# Definiamo la "griglia" degli iperparametri da testare.
# Il dizionario ha come chiavi i nomi dei parametri del modello e come valori le liste di opzioni.
param_grid = {
    # Numero di alberi decisionali da creare nella foresta. Più sono, più il modello è stabile (ma lento).
    'n_estimators': [50, 100, 200],

    # Massima profondità di ogni albero. Limitarla aiuta a prevenire l'overfitting.
    # 'None' significa che gli alberi cresceranno finché non saranno puri.
    'max_depth': [3, 5, 10, None],

    # Il numero minimo di campioni richiesti per dividere un nodo interno dell'albero.
    'min_samples_split': [2, 5]
}

# Calcolo matematico combinatorio:
# Abbiamo 3 opzioni per n_estimators, 4 per max_depth e 2 per min_samples_split.
# Combinazioni totali: 3 * 4 * 2 = 24 combinazioni di iperparametri.
# Poiché useremo una Cross-Validation a 5 fold (cv=5), ogni combinazione verrà testata 5 volte.
# Calcolo finale: 24 * 5 = 120 modelli addestrati in totale! Sklearn gestisce tutto da solo 🚀

# ==========================================
# INIZIALIZZAZIONE DEL MODELLO BASE
# ==========================================

# Creiamo l'istanza "base" del Random Forest.
# Non impostiamo qui i parametri interni perché verranno sovrascritti dalla Grid Search.
# Fissiamo solo il random_state per garantire la riproducibilità dei risultati.
rf = RandomForestClassifier(random_state=42)

# ==========================================
# CONFIGURAZIONE DI GRIDSEARCHCV
# ==========================================

# Inizializziamo l'oggetto GridSearchCV che si occuperà di testare la griglia in modo automatizzato.
grid = GridSearchCV(
    estimator=rf,                  # Il modello base su cui effettuare la ricerca.
    param_grid=param_grid,         # Il dizionario contenente le combinazioni da provare.
    cv=5,                          # Applica una 5-Fold Cross-Validation per ogni singola combinazione.
    scoring='accuracy',            # La metrica usata per decidere qual è la combinazione "migliore".
    n_jobs=-1                      # Sfrutta il parallelismo: '-1' dice a Python di usare TUTTI i core
                                   # della CPU disponibili, velocizzando drasticamente i tempi di calcolo.
)

# ==========================================
# ADDESTRAMENTO E DETERMINAZIONE DEI RISULTATI
# ==========================================

# Avviamo il processo di ricerca. In questa riga di codice scikit-learn:
#  1. Genera le 24 combinazioni.
#  2. Addestra e testa 120 modelli tramite Cross-Validation.
#  3. Identifica la combinazione vincente.
#  4. Riaddestra automaticamente il modello finale con i parametri migliori su TUTTO il dataset (X, y).
grid.fit(X, y)

# Mostriamo a schermo i parametri che hanno ottenuto il punteggio medio più alto nella Cross-Validation.
print(f'Migliori iperparametri: {grid.best_params_}')

# Mostriamo la media dell'accuratezza ottenuta dalla combinazione vincente nei 5 fold.
print(f'Miglior accuratezza:    {grid.best_score_:.4f}')

In [ ]:
# ==========================================
# ESTRAZIONE E ANALISI DEI RISULTATI DELLA GRID SEARCH
# ==========================================

# Convertiamo l'attributo 'cv_results_' (che nativamente è un dizionario Python complesso)
# in un DataFrame di Pandas. In questo modo trasformiamo i dati dei 120 modelli addestrati
# in una comoda tabella pronta per essere manipolata.
risultati = pd.DataFrame(grid.cv_results_)

# Definiamo una lista con i nomi delle colonne che ci interessa analizzare.
# La tabella originale contiene molte colonne tecniche (come i tempi di calcolo);
# noi selezioniamo solo gli iperparametri testati, la media dell'accuratezza e la sua deviazione standard.
colonne = ['param_n_estimators', 'param_max_depth', 'param_min_samples_split', 'mean_test_score', 'std_test_score']

# Eseguiamo tre operazioni fondamentali in un'unica riga di codice (Method Chaining):
#  1. risultati[colonne]: Filtra la tabella tenendo solo le colonne selezionate.
#  2. .sort_values(..., ascending=False): Ordina i modelli in base al punteggio medio
#     dal più alto al più basso (dal migliore al peggiore).
#  3. .head(10): Estrae e mostra solo le prime 10 righe, mostrandoci di fatto la "Top 10"
#     delle migliori configurazioni trovate.
risultati[colonne].sort_values('mean_test_score', ascending=False).head(10)

💡 **Il modello finale è già pronto!** Dopo `grid.fit()`, sklearn ri-allena automaticamente il modello migliore su **tutti** i dati. Lo trovi in `grid.best_estimator_`.

### ⚠️ Attenzione al data leakage
Se devi normalizzare i dati (StandardScaler) e fare cross-validation, NON normalizzare prima! Altrimenti il modello "vede" il test set durante l'allenamento (data leakage). La soluzione è la **Pipeline**.

## 5. Pipeline: incatenare preprocessing + modello

Una `Pipeline` è una catena di passaggi che si applicano in ordine. Il vantaggio? La cross-validation sa che lo scaling va fatto **dentro** ogni fold, non prima.

![pipeline](Media/08/pipeline.png)


In [ ]:
# ==========================================
# CREAZIONE DELLA PIPELINE (FILIERA DI LAVORO)
# ==========================================

# Creiamo una Pipeline. Questo strumento incatena più passaggi in un unico oggetto.
# È fondamentale perché garantisce che la trasformazione dei dati (lo scaling)
# avvenga correttamente all'interno di ogni singolo fold della Cross-Validation,
# evitando il fenomeno del "Data Leakage" (la contaminazione dei dati di test con quelli di train).
pipeline = Pipeline([
    # Passo 1: Standardizzazione delle feature. Le SVM sono estremamente sensibili alla scala dei dati;
    # StandardScaler porta la media a 0 e la varianza a 1 per ogni colonna.
    ('scaler', StandardScaler()),

    # Passo 2: Il modello di classificazione vero e proprio (Support Vector Classifier).
    ('svm', SVC())
])

# ==========================================
# DEFINIZIONE DELLA GRIGLIA DI IPERPARAMETRI
# ==========================================

# Quando si usa una Pipeline, per dire a GridSearchCV quale parametro appartiene a quale modello,
# si usa la sintassi speciale: 'nome_del_passo__nome_del_parametro' (con DOPPIO underscore).
param_grid = {
    # C controlla la regolarizzazione: valori piccoli creano un margine più ampio (più tolleranza agli errori),
    # valori grandi cercano di classificare correttamente tutti i punti di train (rischio overfitting).
    'svm__C': [0.1, 1, 10, 100],

    # Il tipo di kernel da usare: 'linear' per separazioni lineari classiche,
    # 'rbf' (Radial Basis Function) per catturare relazioni non lineari complesse.
    'svm__kernel': ['linear', 'rbf'],

    # Gamma definisce il raggio d'influenza di un singolo esempio di addestramento (solo per kernel RBF).
    # 'scale' e 'auto' sono euristiche automatiche basate sul numero di feature.
    'svm__gamma': ['scale', 'auto']
}

# Combinazioni: 4 (C) * 2 (kernel) * 2 (gamma) = 16 combinazioni.
# Con cv=5 verranno addestrate ed eseguite 16 * 5 = 80 pipeline complete.

# ==========================================
# CONFIGURAZIONE ED ESECUZIONE DELLA GRID SEARCH
# ==========================================

# Inizializziamo la Grid Search passando la pipeline come stimatore principale.
# Internamente, per ogni combinazione e per ogni fold, Sklearn prenderà i dati,
# applicherà lo StandardScaler e poi allenerà la SVM.
grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Avviamo l'intera fiera del calcolo automatico sul dataset.
grid.fit(X, y)

# ==========================================
# VISUALIZZAZIONE DEI RISULTATI
# ==========================================

# Stampiamo la combinazione di parametri che ha ottenuto le performance migliori.
print(f'Migliore configurazione: {grid.best_params_}')

# Stampiamo il punteggio medio di accuratezza ottenuto dalla configurazione vincente.
print(f'Accuratezza:             {grid.best_score_:.4f}')

## 6. Workflow completo: il modo "giusto" di fare ML
![wf](Media/08/wf.png)

Ecco lo schema che dovresti applicare in QUALSIASI progetto ML:

In [ ]:
# ────────────────────────────────────────────────────────────────
# WORKFLOW COMPLETO (Riferimento per i tuoi progetti futuri)
# ────────────────────────────────────────────────────────────────

# ==========================================
# STEP 1: ISOLAMENTO DEL TEST SET FINALE
# ==========================================

# Separiamo IMMEDIATAMENTE una parte di dati (20%) che diventerà il nostro "Test Set".
# Questa porzione è SACRA: non deve subire trasformazioni, non deve essere vista dalla Grid Search
# e non deve influenzare la scelta degli iperparametri. Servirà solo alla fine per il "voto reale".
# Parametri cruciali:
#  - stratify=y: assicura che le percentuali delle classi (es. 60% sani, 40% malati)
#    siano identiche sia in X_dev che in X_test.
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ==========================================
# STEP 2: COSTRUZIONE DELLA PIPELINE
# ==========================================

# Definiamo la catena di montaggio. Ogni volta che invocheremo il fit su questa pipeline,
# i dati verranno prima standardizzati dallo 'scaler' e poi passati al classificatore 'clf'.
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=10000))
])

# ==========================================
# STEP 3: OTTIMIZZAZIONE SOLO SUL DEV SET (CV)
# ==========================================

# Definiamo lo spazio di ricerca per il parametro di regolarizzazione 'C' della regressione logistica.
param_grid = {
    'clf__C': [0.01, 0.1, 1, 10, 100]
}

# Configuriamo la Grid Search passandogli la pipeline e la griglia di parametri.
# Nota fondamentale: facciamo il fit SOLO su X_dev e y_dev!
# La Cross-Validation a 5 fold simulerà internamente la divisione tra train e validation,
# permettendoci di trovare il parametro 'C' migliore senza mai toccare il test set finale.
grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_dev, y_dev)

# Stampiamo i risultati della ricerca automatica sul set di sviluppo.
print(f'Migliore C: {grid.best_params_}')
print(f'Accuratezza CV: {grid.best_score_:.4f}')

# ==========================================
# STEP 4: VALUTAZIONE FINALE SUL TEST SET
# ==========================================

# Estraiamo il modello vincitore (quello riaddestrato su tutto X_dev con il 'C' ottimale).
miglior_modello = grid.best_estimator_

# Usiamo il modello finale per fare le predizioni sul test set (i dati tenuti sotto chiave nello Step 1).
# Questo passaggio avviene una sola volta nella vita del progetto!
predizioni = miglior_modello.predict(X_test)

# Calcoliamo l'accuratezza finale sul test set.
print(f'\nAccuratezza sul test set finale: {accuracy_score(y_test, predizioni):.4f}')

# Generiamo il report di classificazione completo che mostra:
#  - Precision: quanti dei positivi predetti sono reali?
#  - Recall: quanti dei positivi reali siamo riusciti a catturare?
#  - F1-Score: la media armonica tra le due metriche precedenti.
# Passiamo 'target_names' per vedere i nomi reali delle classi ("malignant", "benign") al posto di 0 e 1.
print('\nReport completo:')
print(classification_report(y_test, predizioni, target_names=data.target_names))

💡 **Perché tenere un test set "intoccato"?**
Se usi i dati di test per scegliere il modello migliore, alla fine non sai più se quel numero è onesto. Il test set finale è la tua "prova del nove" - lo guardi UNA volta sola, alla fine.

Schema mentale:
```
Tutti i dati
    ├── Train+Validation (80%) ← qui faccio cross-validation e tuning
    └── Test (20%)              ← guardato UNA VOLTA, alla fine
```

## 7. Quale metrica scegliere?

Una checklist veloce:

| Tipo di problema | Metrica suggerita | Quando |
|------------------|-------------------|--------|
| Classificazione bilanciata | `accuracy` | Le classi sono ~50/50 |
| Classificazione sbilanciata | `f1`, `roc_auc` | Una classe rara (es. frodi) |
| Frodi/diagnosi mediche | `recall` | Meglio falsi allarmi che mancare un caso |
| Spam/raccomandazioni | `precision` | Meglio mancarne uno che dare un falso positivo |
| Regressione | `neg_root_mean_squared_error`, `r2` | A seconda di cosa vuoi minimizzare |

Cambia `scoring='...'` in `cross_val_score` o `GridSearchCV` per usare un'altra metrica.

![error](Media/08/error.png)


## 🎓 Riepilogo del corso completo

Hai percorso un viaggio importante! Ecco cosa hai imparato:

| # | Notebook | Concetti chiave |
|---|----------|----------------|
| 1 | Matematica base | Vettori, matrici, derivate, gradiente, statistica |
| 2 | Dati | Pandas, pulizia, encoding, scaling, split |
| 3 | Regressione lineare | MSE, gradient descent, R² |
| 4 | Regressione logistica | Sigmoide, soglia, precision/recall/F1, ROC |
| 5 | Alberi e Random Forest | Pruning, ensemble, feature importance |
| 6 | K-Means | Clustering non supervisionato, gomito, silhouette |
| 7 | PCA | Riduzione dimensionalità, varianza spiegata |
| 8 | Valutazione e tuning | CV, GridSearch, Pipeline, workflow |

## 🗺️ Roadmap: dove andare adesso

### Livello successivo (subito dopo questo corso):
1. **Pratica su Kaggle**: prova competizioni "Getting Started" come *Titanic* o *House Prices*
2. **Approfondisci sklearn**: esplora altri modelli — Gradient Boosting, XGBoost, LightGBM
3. **Feature engineering**: l'arte di creare nuove colonne dai dati esistenti (spesso più importante del modello!)
4. **Imbalanced learning**: SMOTE, class_weight per classi sbilanciate

### Strada del Deep Learning:
1. **Reti neurali base** con PyTorch o Keras/TensorFlow
2. **CNN** per immagini (computer vision)
3. **Transformer** per testo (NLP, LLM)

### Strada "applicazioni reali":
1. **MLOps**: come mettere un modello in produzione (Docker, FastAPI, MLflow)
2. **Time series**: previsioni temporali (ARIMA, Prophet)
3. **Recommender systems**: come fa Netflix/Spotify

### Risorse consigliate (gratuite):
- 📚 *"Hands-On Machine Learning"* di Aurélien Géron (libro - il più consigliato in assoluto)
- 🎥 *Andrew Ng - Machine Learning Specialization* su Coursera
- 🎥 *StatQuest* su YouTube (spiegazioni intuitive eccezionali)
- 📖 La documentazione di sklearn: scikit-learn.org/stable/user_guide.html
- 🏆 Kaggle Learn: kaggle.com/learn (mini-corsi gratis, molto pratici)

## 💡 Consiglio finale

Il ML si impara **con le mani sporche**. Scegli un dataset che ti interessa davvero (sport, musica, videogiochi, finanza, ...) e prova ad applicare quello che hai imparato. I primi 3-4 progetti saranno frustranti, poi qualcosa scatta. 🚀

**In bocca al lupo per il tuo viaggio nel ML! 🍀**